In [ ]:
import csv

def calculate_avg(movies, score_type):
    """
    This function calculates the average score based off of the parameter passed and returns a
    float value rounded off to one decimal point.

    Parameters:
        movies (list): A list of dictionaries containing all the movie data
        score_type (string): The type of score (e.g., "imdbRating", "Jump Scare Rating") to calculate
        the average for.

    Returns:
        float: A float value indicating the average value, rounded to one decimal point
    """

    sum_score = 0
    for movie in movies:
        score = get_value(movie, score_type)
        if score:
            sum_score = float(movie[score_type])
    return round(sum_score / len(movies), 1)


def clean_movie_data(movie):
    """
    Clean the movie data by converting the 'Runtime' value to an integer. If the 'Runtime' value
    is not present, the function will call the < convert_to_int() > function.

    Parameters:
        movie (dict): dictionary containing key-value pairs representing data for one movie

    Returns:
        dict: dictionary containing key-value pairs representing data for one movie
    """

    try:
        movie["Runtime"] = convert_to_int(movie["Runtime"].split()[0])
    except KeyError:
        movie = convert_movie_fields_to_int(movie)
    return convert_movie_fields_to_int(movie)


def convert_to_int(value):
    """
    Attempts to convert a string, number, or boolean value to an int.
    If a ValueError exception is encountered, the function returns the value unchanged.

    Parameters:
        value (str|bool): a string or boolean value to be converted

    Returns:
        int: if the value is successfully converted else returns value unchanged
    """
    try:
        return int(value)
    except ValueError:
        return value


def convert_movie_fields_to_int(movie):
    """
    Loops through a movie dictionary's key-value pairs and converts values
    that can be converted to an integer using the convert_to_int() function.

    Parameters:
        movie (dict): dictionary containing key-value pairs representing data for one movie

    Returns:
        dict: dictionary containing cleaned key-value pairs for the movie
    """
    for k, v in movie.items():
        movie[k] = convert_to_int(v)
    return movie


def count_movie_by_rating(movies, rating):
    """
    Loops through movies (list) and check if rating (str) is included in the dictionary key: 'Rated'.
    Increment the count by 1 if the given rating matches the movie's rating.

    Parameters:
        movies (list): List of dictionaries representing all movies
        rating (str): Audience Rating of the movie to check for

    Returns:
        int: Number of movies that have the supplied Rating
    """

    movie_count = 0
    for movie in movies:
        if movie["Rated"].lower() == rating.lower():
            movie_count += 1
    return movie_count


def filter_movie_by_genre(movies, genre):
    """
    Loops through movies (list) and check if genre (str) is included in the dictionary key: 'Genre'.
    Append the dictionary and all of its data to an empty list and return the final list.

    Parameters:
        movies (list): List of dictionaries representing all movies
        genre (str): Movie genre to check for

    Returns:
        list: List with movie dictionaries that include the supplied genre
    """

    movie_list = []
    for movie in movies:
        if genre.lower() in movie["Genre"].lower():
            movie_list.append(movie)
    return movie_list


def get_jumpscares(jumpscares, movie_title):
    """
    Loops through jumpscare data (list) and check to see if the value of the key 'Movie Name'
    matches the supplied movie_title (str) and returns the values of the keys 'Jump Count'
    and 'Jump Scare Rating' as a tuple

    Parameters:
        jumpscares (list): information about the movie's jump scares
        movie_title (str): value of the key 'Title' from a movie dictionary

    Returns:
        tuple: A tuple with the first value representing a jump count and the second value
        representing a jump scare rating. Both of these tuples should be None if the
        movie name does not exist in the jumpscares list.
    """

    for item in jumpscares:
        if item["Movie Name"].lower().strip() == movie_title.lower().strip():
            return item["Jump Count"], item["Jump Scare Rating"]
    return None, None


def get_value(movie, key_to_check):
    """
    This function checks if a parameter value exists or not in the dictionary, if it does it
    returns the value, else it returns False

    Parameters:
        movie (dict): a dictionary containing all the data about one movie
        key_to_check (string): a parameter of the dictionary

    Returns:
        value : if the value exists, return the value of the parameter. can be a string, float
        or int based on which parameter is being passed.
        None: if the value does not exist, return None
    """

    if movie[key_to_check]:
        return movie[key_to_check]
    return None


def read_csv_to_dicts(filepath, encoding="utf-8", newline="", delimiter=","):
    """
    Accepts a file path for a .csv file to be read, creates a file object,
    and uses csv.DictReader() to return a list of dictionaries
    that represent the row values from the file.

    Parameters:
        filepath (str): path to csv file
        encoding (str): name of encoding used to decode the file
        newline (str): specifies replacement value for newline '\n'
        or '\r\n' (Windows) character sequences
        delimiter (str): delimiter that separates the row values

    Returns:
        list: nested dictionaries representing the file contents
    """

    with open(filepath, "r", newline=newline, encoding=encoding) as file_obj:
        data = []
        reader = csv.DictReader(file_obj, delimiter=delimiter)
        for line in reader:
            data.append(line)
    return data



def write_dicts_to_csv(filepath, data, fieldnames, encoding="utf-8", newline=""):
    """
    Uses csv.DictWriter() to write a list of dictionaries to a target CSV file as row data.
    The passed in fieldnames list is used by the DictWriter() to determine the order
    in which each dictionary's key-value pairs are written to the row.

    Parameters:
        filepath (str): path to target file (if file does not exist it will be created)
        data (list): dictionary content to be written to the target file
        fieldnames (seq): sequence specifying order in which key-value pairs are written to each row
        encoding (str): name of encoding used to encode the file
        newline (str): specifies replacement value for newline '\n'
        or '\r\n' (Windows) character sequences.

    Returns:
        None
    """

    with open(filepath, "w", encoding=encoding, newline=newline) as file_obj:
        writer = csv.DictWriter(file_obj, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)


Load in the horror movie and jumpscare datasets. 

In [ ]:
# Load in data
filepath_movies = "./data/horror_movies.csv"
filepath_jumpscare = "./data/movie_jumpscares.csv"
horror_movies = read_csv_to_dicts(filepath_movies)
jumpscares = read_csv_to_dicts(filepath_jumpscare)

Run the data cleaning steps and print out sample elements for inspection.

In [ ]:
# Clean data
for movie in horror_movies:
    clean_movie_data(movie)

for data in jumpscares:
    clean_movie_data(data)

print(f"First element in horror_movies: {horror_movies[0]}")
print(f"First element in jumpscares: {jumpscares[0]}")


Filter the horror movies dataset so that we only include movies that have "horror" listed in their genre. This is an extra quality control step.

In [ ]:
horror_movies = filter_movie_by_genre(horror_movies, "horror")
print(f"\nLength of Horror movies \n{len(horror_movies)}")


How many movies are there with each rating?

In [ ]:
# Print out how many movies have each rating
unique_ratings = ["PG", "PG-13", "R", "Not Rated", "Passed", "Unrated"]

movie_ratings_count = {}
for rating in unique_ratings:
    movie_ratings_count[rating] = count_movie_by_rating(horror_movies, rating)

print(f"Count of horror movies in each viewer ratings:\n{movie_ratings_count}")


How many movies contain a jumpscare?

In [ ]:

for movie in horror_movies:
    movie["Jumpscare_count"], movie["Jumpscare_rating"] = get_jumpscares(
        jumpscares, movie["Title"]
    )
print(f"\nLength of horror movies with jump scare data:\n{len(horror_movies)}")


What is the average IMDb movie rating and jumpscare rating for these movies?

In [ ]:
avg_imdb_rating = calculate_avg(horror_movies, "imdbRating")
avg_jumpscare_rating = calculate_avg(horror_movies, "Jumpscare_rating")

print(f"The average imdb Score is: {avg_imdb_rating}")
print(f"The average jumpscare Score is: {avg_jumpscare_rating}")


Which movies are the highest rated, based on IMDb and jumpscare rating?

In [ ]:
high_rated_movies = []

for movie in horror_movies:
    imdb_rating = get_value(movie, "imdbRating")
    jumpscare_rating = get_value(movie, "Jumpscare_rating")

    if imdb_rating is not None and jumpscare_rating is not None:
        if (
            float(imdb_rating) > avg_imdb_rating
            and float(jumpscare_rating) > avg_jumpscare_rating
        ):
            high_rated_movies.append({
                "Title": get_value(movie, "Title"),
                "IMDb_rating": imdb_rating,
                "Jumpscare_rating": jumpscare_rating,
            })

print("High-rated movies:")
for movie in high_rated_movies:
    print(movie)